# InternViT patch alignment: real-data pilot

RPA-inspired, **not an official RPA reproduction**. This notebook adds a native L28 spatial adapter to the existing EEG architecture. It compares the unchanged route_c architecture, L28 mean control, and L28 patch adapter under one fixed protocol. Historical 78.8% used a different split/normalization/batch/scheduler; it is not the expected accuracy of this corrected pilot.

Start with one subject, 200 training concepts, 5 epochs, seed 0; retain the full 200-way test gallery. No text supervision or SCORE. Full runs follow only after this pilot succeeds. Choose an L4 24 GB or A100 and High RAM. See `docs/PATCH_EXPERIMENT.md` for storage and memory estimates.

Existing notebooks/results are never written. Each experiment has a new directory. Pin the code commit and Hugging Face revision before running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys
REPO = Path('/content/eeg-patch-pr2')
BRANCH = 'audit/phase1-reproducible-alignment'
if not REPO.exists():
    subprocess.run(['git','clone','--branch',BRANCH,'https://github.com/firozehrazavi/eeg-alignment-phase1.git',str(REPO)],check=True)
print('Code commit:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())
# Do not silently pull another revision into a running experiment.
subprocess.run([sys.executable,'-m','pip','install','-q','torch==2.5.1','torchvision==0.20.1',
                '--index-url','https://download.pytorch.org/whl/cu121'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-patch-colab.txt')],check=True)
print('Setup complete. Restart the runtime now if torch/numpy were already imported, then continue below.')

In [ ]:
# Run this after any runtime restart.
from pathlib import Path
import subprocess, sys, json, os, shutil
REPO = Path('/content/eeg-patch-pr2')
os.chdir(REPO)
subprocess.run([sys.executable,'-m','unittest','discover','-s','tests','-v'],check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before real-data extraction.'
print(torch.__version__,torch.cuda.get_device_name(),torch.cuda.get_device_properties(0).total_memory/2**30,'GiB')

## Explicit input preflight

Put the authoritative image tree and distributed EEG under `RAW`. Put the two existing metadata/pooled caches under `CACHE`. `_z16.npy` files are deliberately not used. Distributed preprocessed EEG is labeled **benchmark_preprocessed**: upstream whitening has not been established to satisfy strict inductive LOSO. Do not turn on strict mode by changing a label alone.

The listed locations are required inputs, not evidence that files exist in your Drive. The converter verifies metadata row ordering and hashes every image. Use only trusted pickle-based files.

In [ ]:
RAW = Path('/content/drive/MyDrive/THINGS-EEG2')
CACHE = Path('/content/drive/MyDrive/eeg_results/cache')
WORK = Path('/content/drive/MyDrive/eeg_results/patch_phase1')
PREP = WORK/'pilot_sub01_200concepts_v1'
SUBJECTS = ['sub-01']
required = [RAW/'images_set/image_metadata.npy', CACHE/'meta_v6.pt', CACHE/'feat6_internvit6b.pt']
required += [RAW/f'Preprocessed_data_250Hz/{s}/preprocessed_eeg_{part}.npy'
             for s in SUBJECTS for part in ['training','test']]
missing = [str(p) for p in required if not p.is_file()]
print('Missing input files:\n'+'\n'.join(missing) if missing else 'Required input containers found.')
assert not missing, 'Upload/mount the exact files above; no synthetic fallback is used.'
if not PREP.exists():
    subprocess.run([sys.executable,'-m','phase1.patch_prepare','--root',str(RAW),'--cache',str(CACHE),
                    '--output',str(PREP),'--subjects',*SUBJECTS,'--concepts','200'],check=True)
else:
    assert (PREP/'dataset.json').is_file(), 'Incomplete preparation: choose a new PREP directory.'
print('Prepared manifest:',PREP/'dataset.json')

## Extract one layer once

The full native 32×32×3200 FP16 cache is roughly 102.2 GiB for 16,740 images. This 2,200-image pilot is roughly 13.4 GiB. These estimates assume the expected native grid; the extractor checks actual shapes. Extract batch 1 initially, no gradients, BF16 backbone, FP16 disk cache, discard CLS, hook only layer 28 and truncate later layers. Do not reduce token resolution or quantize for this first comparison.

The first invocation resolves a model revision and saves it; subsequent invocations reuse it. Recording a revision does not prove it matches the historical cache. Sampled L28 means must agree before training. Cache extraction resumes incomplete rows. Drive mmap writes may be slow, but survive runtime loss; stage locally for training.

In [ ]:
from huggingface_hub import model_info
WORK.mkdir(parents=True,exist_ok=True)
revision_file = PREP/'internvit_revision.txt'
if not revision_file.exists():
    revision_file.write_text(model_info('OpenGVLab/InternViT-6B-448px-V2_5').sha)
revision = revision_file.read_text().strip()
print('Pinned model revision:',revision)
subprocess.run([sys.executable,'-m','phase1.patch_cache','--catalog',str(PREP/'catalog.json'),
                '--output',str(PREP/'tokens_L28'),'--revision',revision,'--layer','28','--batch','1'],check=True)
# Extraction ran in a separate process, releasing backbone VRAM before training.
LOCAL = Path('/content/patch_pilot_data')
if not LOCAL.exists():
    shutil.copytree(PREP,LOCAL)
else:
    raise FileExistsError('Use a fresh LOCAL directory to avoid stale staged caches.')
print('Staged:',LOCAL)

In [ ]:
# NEW result directory; changing any protocol requires a new name.
OUT = WORK/'pilot_sub01_200concepts_run1'
args = [sys.executable,'-m','phase1.patch_experiment','--dataset',str(LOCAL/'dataset.json'),
        '--config',str(REPO/'configs/patch_pilot.json'),'--output',str(OUT)]
# For an interrupted identical run only: args += ['--resume']
subprocess.run(args,check=True)
print((OUT/'results.csv').read_text())
print('Measured peak training VRAM GiB:',json.loads((OUT/'provenance.json').read_text())['peak_gpu_allocated_bytes']/2**30)
subprocess.run([sys.executable,'-m','pip','freeze'],stdout=open(OUT/'environment.txt','w'),check=True)

## Full evaluation after the pilot

Prepare a **new** dataset using all ten subject IDs and `--concepts 0`; extract the same one-layer cache once and reuse it across subjects/folds/seeds. Run `patch_full_within_subject_diagonal.json` first, then `patch_full_loso_diagonal.json`. Each schedules all available subjects and seeds 0/1/2. With ten subjects, LOSO is 30 runs per arm, each trained on the other nine subjects. Keep all ten in the manifest even when `targets` selects a pilot fold.

Use the `multi_positive` configs as a separate objective ablation with otherwise identical settings. The strict LOSO config refuses benchmark-preprocessed manifests; supply audited target-independent preprocessing with written `audit_evidence` first. Hyperparameters are fixed a priori: do not select them using outer target results. Any later tuning needs source-subject inner folds.

`--resume` skips completed arms only after checking config, catalog, code and recorded input hashes. An interrupted arm restarts from epoch 1 with the same seed; it does not resume optimizer state. Save each protocol/objective under a unique output directory. CSLS gets a distinct transductive row; no SCORE recovery or calibration is mixed into zero-calibration accuracy. No text supervision is included in this visual experiment.

The pilot establishes executability and resource use, not SOTA. Report 200-way Top-1/Top-5, average seeds within subjects before averaging subjects, and retain per-subject/per-seed outputs. Do not compare the pilot directly with published full-data accuracy.